In [1]:
import pandas as pd
df = pd.read_csv(r'D:\DataAnalyticsProjects\ecommerce-retail-analysis\data\tableau\unified_tableau.csv')

# Check Champions count
champions = df[df['segment'] == 'Champions']
print("Champions total rows:", len(champions))
print("Champions unique customers:", champions['customer_unique_id'].nunique())

# Check all segments
print("\nUnique customers per segment:")
print(df.groupby('segment')['customer_unique_id'].nunique().sort_values(ascending=False))

Champions total rows: 30566
Champions unique customers: 23246

Unique customers per segment:
segment
Champions              23246
Loyal Customers        11848
Lost                   11736
At Risk                11639
Potential Loyalists    11625
Cannot Lose Them       11266
Hibernating             6120
Needs Attention         5960
Name: customer_unique_id, dtype: int64


In [2]:
import pandas as pd

# Load original rfm export
rfm_orig = pd.read_csv(r'D:\DataAnalyticsProjects\ecommerce-retail-analysis\data\tableau\rfm_segments.csv')

# Load unified
unified = pd.read_csv(r'D:\DataAnalyticsProjects\ecommerce-retail-analysis\data\tableau\unified_tableau.csv')

print("=== ORIGINAL rfm_segments.csv ===")
print(rfm_orig.groupby('segment')['customer_unique_id'].nunique().sort_values(ascending=False))
print(f"Total unique customers: {rfm_orig['customer_unique_id'].nunique():,}")

print("\n=== UNIFIED TABLE ===")
print(unified.groupby('segment')['customer_unique_id'].nunique().sort_values(ascending=False))
print(f"Total unique customers: {unified['customer_unique_id'].nunique():,}")

print("\n=== OVERLAP CHECK ===")
# Pick one customer from Champions in original — find them in unified
orig_champions = set(rfm_orig[rfm_orig['segment']=='Champions']['customer_unique_id'])
unified_champions = set(unified[unified['segment']=='Champions']['customer_unique_id'])

print(f"Original Champions: {len(orig_champions):,}")
print(f"Unified Champions:  {len(unified_champions):,}")
print(f"In original but not unified: {len(orig_champions - unified_champions):,}")
print(f"In unified but not original: {len(unified_champions - orig_champions):,}")

# Check a specific customer that appears in unified Champions but not original
extra = list(unified_champions - orig_champions)[:3]
print(f"\nSample customers in unified Champions but not original Champions:")
for cid in extra:
    print(f"\nCustomer: {cid}")
    print("In original rfm:", rfm_orig[rfm_orig['customer_unique_id']==cid][['customer_unique_id','segment']].to_string())
    print("In unified:", unified[unified['customer_unique_id']==cid][['customer_unique_id','segment']].head(2).to_string())

=== ORIGINAL rfm_segments.csv ===
segment
Champions              23439
Lost                   11748
Loyal Customers        11667
Potential Loyalists    11613
At Risk                11596
Cannot Lose Them       11325
Hibernating             6031
Needs Attention         6021
Name: customer_unique_id, dtype: int64
Total unique customers: 93,357

=== UNIFIED TABLE ===
segment
Champions              23246
Loyal Customers        11848
Lost                   11736
At Risk                11639
Potential Loyalists    11625
Cannot Lose Them       11266
Hibernating             6120
Needs Attention         5960
Name: customer_unique_id, dtype: int64
Total unique customers: 93,357

=== OVERLAP CHECK ===
Original Champions: 23,439
Unified Champions:  23,246
In original but not unified: 611
In unified but not original: 418

Sample customers in unified Champions but not original Champions:

Customer: 78444f8aaffc4cb5acd57733fe5dc3eb
In original rfm:                      customer_unique_id          seg

In [3]:
import psycopg2, os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(r'D:\DataAnalyticsProjects\ecommerce-retail-analysis\.env')

conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)

# Get segment counts directly from the view
q = """
SELECT segment, COUNT(DISTINCT customer_unique_id) as customers
FROM olist.v_rfm_scores
GROUP BY segment
ORDER BY customers DESC
"""
view_counts = pd.read_sql(q, conn)
print("=== v_rfm_scores VIEW ===")
print(view_counts)

# Check the specific customer
q2 = """
SELECT customer_unique_id, segment, r_score, f_score, m_score, rfm_total
FROM olist.v_rfm_scores
WHERE customer_unique_id = 'e6720a28057c2e7ab8028be970af2f82'
"""
print("\n=== Specific customer in VIEW ===")
print(pd.read_sql(q2, conn))
conn.close()

C:\Users\CHANDAN RAM SAII G\AppData\Local\Temp\ipykernel_10892\283614295.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  view_counts = pd.read_sql(q, conn)


=== v_rfm_scores VIEW ===
               segment  customers
0            Champions      23246
1      Loyal Customers      11848
2                 Lost      11736
3              At Risk      11639
4  Potential Loyalists      11625
5     Cannot Lose Them      11266
6          Hibernating       6120
7      Needs Attention       5960

=== Specific customer in VIEW ===


C:\Users\CHANDAN RAM SAII G\AppData\Local\Temp\ipykernel_10892\283614295.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(q2, conn))


                 customer_unique_id    segment  r_score  f_score  m_score  \
0  e6720a28057c2e7ab8028be970af2f82  Champions        4        3        3   

   rfm_total  
0         10  
